<a href="https://colab.research.google.com/github/naweabaasa/nlp-project-group-Raidah-and-Abaasa/blob/main/NLP_SHIPPING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets huggingface_hub gradio fastapi uvicorn pypdf sentence-transformers

In [2]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: Abaasa



LOADING THE SENTIMENT MODEL

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

LOCAL_MODEL_DIR = "/content/drive/MyDrive/model_1"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Classifying a negative and a positive sentence
print("Negative sentence:", clf("Terrible experience, I want my money back."))
print("Positive sentence:", clf("Absolutely wonderful service, I loved it!"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Negative sentence: [{'label': 'NEGATIVE', 'score': 0.9936692118644714}]
Positive sentence: [{'label': 'POSITIVE', 'score': 0.9991967082023621}]


Uploading the model to Hugging Face

In [ ]:

MODEL_NAME = "Raidah_Nawe_sentiment_model"
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

model.push_to_hub(MODEL_ID)
tokenizer.push_to_hub(MODEL_ID)
print("Model page:", f"https://huggingface.co/{MODEL_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xi9ik8w/model.safetensors:   6%|5         | 15.9MB /  268MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model page: https://huggingface.co/Abaasa/Raidah_Nawe_sentiment_model


Writing a model card

In [11]:
from huggingface_hub import ModelCard


MODEL_NAME = "Raidah_Nawe_sentiment_model"
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

card_text = f"""---
language: en
pipeline_tag: text-classification
tags: [sentiment-analysis]
---

# {MODEL_NAME}

Sentiment classifier (positive / negative) fine-tuned from `distilbert-base-uncased`.

**Group members:**
Ntale Raidah,
Nawe Abaasa

**Dataset:** stanfordnlp/imdb(movie reviews)

## Results on the test set
| Metric | Score |
|---|---|
| Accuracy | 0.9890 |
| F1 | 0.9886 |

## Example predictions
Terrible experience, I want my money back.
Absolutely wonderful service, I loved it!
It hurt to watch this movie

## Limitations
The model is fine-tuned primarily on movie reviews (IMDB dataset) and may not generalize well to other domains such as customer support, sarcasm, or non-English languages like Luganda.
"""

ModelCard(card_text).push_to_hub(MODEL_ID)
print("Model card uploaded.")

Model card uploaded.


In [12]:
# Loading it back from the Hub
hub_clf = pipeline("text-classification", model=MODEL_ID)
hub_clf("I really enjoyed this class!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9989396929740906}]

In [13]:
# Building a Gradio demo
import gradio as gr

def predict(text):
    if not text.strip():
        return {}
    result = hub_clf(text)[0]
    score = result["score"]
    # Gradio's Label component shows a confidence bar for each class
    other = "NEGATIVE" if result["label"] == "POSITIVE" else "POSITIVE"
    return {result["label"]: score, other: 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    description="Type a sentence and see if it is positive or negative.",
    examples=["The food was amazing!", "Worst service ever.", "It was okay, nothing special."],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17fb2ccdf4d37bd35d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# Deploying the demo to Hugging Face Spaces
%%writefile app.py
import gradio as gr
from transformers import pipeline

MODEL_ID = "Abaasa/Raidah_Nawe_sentiment_model"
clf = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {}
    r = clf(text)[0]
    other = "NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"
    return {r["label"]: r["score"], other: 1 - r["score"]}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    examples=["The food was amazing!", "Worst service ever."],
)
demo.launch()

Writing app.py


In [15]:
%%writefile requirements.txt
transformers
torch

Writing requirements.txt


Turning the model into an API with FastAPI

A demo is for people. An API is for other programs to use the model

In [16]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "Abaasa/Raidah_Nawe_sentiment_model"

app = FastAPI(title="Sentiment API")

# Loading the model once when the app starts, not on every request (that would be very slow)
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))

Writing main.py


In [17]:
# Put your model id into main.py, then start the API in the background
#open("main.py", "w").write(open("main.py").read().replace("Abaasa/Raidah_Nawe_sentiment_model", MODEL_ID))

import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)   # give it time to load the model
print("API started")

API started


In [18]:
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

for sentence in ["I love this app!", "This is so frustrating.", ""]:
    r = requests.post(f"{BASE}/predict", json={"text": sentence})
    print(r.status_code, r.json())

{'status': 'ok'}
200 {'label': 'POSITIVE', 'score': 0.9987}
200 {'label': 'NEGATIVE', 'score': 0.9972}
400 {'detail': 'Text cannot be empty'}


In [20]:
server.terminate()   # stopping the API

Building the document chatbot

In [21]:
#  Building a chatbot over the group PDF
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype="auto",
)

def ask_llm(messages, max_tokens=400):
    out = pipe(messages, max_new_tokens=max_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

print(ask_llm([
    {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
    {"role": "user", "content": "What is natural language processing?"},
]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Natural Language Processing (NLP) is the branch of artificial intelligence that enables computers to understand, interpret, and generate human language.


In [23]:
# Comparing
import time, json

test_sentences = [
    "The movie was a long awaited release.",
    "I was pleased to see that she had black hair!",
    "This film has a lot of raw potential.",
    "Like I said its a hidden surprise.",
    "This movie really kicked some ass.",
]

for s in test_sentences:
    t0 = time.time(); mine = hub_clf(s)[0]["label"]; t_mine = time.time() - t0
    t0 = time.time()
    llm_label = ask_llm([
        {"role": "system", "content": "Classify the sentiment. Reply with ONE word: POSITIVE or NEGATIVE."},
        {"role": "user", "content": s},
    ], max_tokens=5).strip()
    t_llm = time.time() - t0
    print(f"{s[:45]:<47} mine={mine:<9} ({t_mine:.2f}s)  Qwen={llm_label:<9} ({t_llm:.2f}s)")

The movie was a long awaited release.           mine=POSITIVE  (0.01s)  Qwen=POSITIVE  (0.31s)
I was pleased to see that she had black hair!   mine=POSITIVE  (0.01s)  Qwen=POSITIVE  (0.20s)
This film has a lot of raw potential.           mine=POSITIVE  (0.01s)  Qwen=POSITIVE  (0.20s)
Like I said its a hidden surprise.              mine=POSITIVE  (0.01s)  Qwen=POSITIVE  (0.21s)
This movie really kicked some ass.              mine=NEGATIVE  (0.01s)  Qwen=POSITIVE  (0.19s)


In [24]:
complaint = "I paid for MTN data bundles yesterday but they never came. This is the third time! Fix it today."

reply = ask_llm([
    {"role": "system", "content": (
        "Extract information from customer complaints. "
        "Reply ONLY with JSON with keys: product, issue, urgency (low, medium or high). No other text."
    )},
    {"role": "user", "content": complaint},
])
print(reply)
data = json.loads(reply.strip().strip("`").removeprefix("json"))
data

{
  "product": "MTN data bundles",
  "issue": "Delivery of purchased products",
  "urgency": "high"
}


{'product': 'MTN data bundles',
 'issue': 'Delivery of purchased products',
 'urgency': 'high'}

Answering questions from the agriculture PDF

In [25]:
# Uploading the pdf
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Using:", PDF_PATH)

Saving agriculture.pdf to agriculture.pdf
Using: agriculture.pdf


In [26]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
print(f"{len(pages)} pages. First 300 characters of page 1:\n")
print(pages[0][1][:300])

20 pages. First 300 characters of page 1:

GUIDE OF BEST FARMING PRACTICES



In [27]:
# Step 1 Splitting into chunks (keeping the page number)
def chunk_pages(pages, chunk_size=400, overlap=80):
    chunks = []
    for page_num, text in pages:
        text = " ".join(text.split())          # clean up extra spaces and line breaks
        start = 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if len(piece) > 50:                # skip tiny leftovers
                chunks.append({"page": page_num, "text": piece})
            start += chunk_size - overlap      # overlap so sentences aren't cut in half
    return chunks

chunks = chunk_pages(pages)
print(len(chunks), "chunks")
print(chunks[0])

64 chunks
{'page': 2, 'text': 'The information published in the foregoing guide does not engage ECHO; it is the sole responsibility of its author.'}


In [28]:
# Step 2: Embed the chunks
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(chunk_vectors.shape)   # (number of chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(64, 384)


In [29]:
# Step 3: Retrieving the most relevant chunks for a question
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec            # cosine similarity (vectors are normalised)
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]

question = "What happens when crops are not rotated?"
for hit in retrieve(question):
    print(f"page {hit['page']}  score={hit['score']:.2f}  {hit['text'][:120]}...")

page 9  score=0.62   to application of fertilizer and other chemical products. • Rotating the crops to take better advantage of labor and eq...
page 9  score=0.52  to satisfy the needs of consumers and the local markets. The land will be better developed, the soil more productive whe...
page 9  score=0.47   grow better and in improved conditions. • Observing established principles and safety rules for operation of equip- men...


In [39]:
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. "
    "Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document."
)

def rag_answer(question, k=3):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)
    return ask_Qwen([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

print(rag_answer(question))

When crops are not rotated, it can lead to several negative outcomes:

1. **Reduced Soil Productivity**: Without rotation, the soil may become depleted of nutrients over time due to continuous use of the same type of crop. This reduces its overall productivity.

2. **Increased Disease and Pest Problems**: Continuous planting of the same crop can create favorable conditions for specific diseases and pests to thrive. For example, monoculture corn fields are often plagued by rootworms because they have no natural predators to keep their populations under control.

3. **Decreased Biological Benefits**: Crop rotations help manage weeds through competition, mechanical methods, biological controls, or herbicides. When crops aren't rotated, these benefits diminish, making it harder to maintain healthy plants without additional inputs such as chemicals.

4. **Lower Yield Potential**: Different crops perform best under varying environmental conditions. By rotating them, farmers can optimize yiel

In [50]:
# Define ask_Qwen as an alias to your ask_llm function to fix the NameError
ask_Qwen = ask_llm

SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. "
    "Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document."
)

def rag_answer_with_source(question, k=3):
    hits = retrieve(question, k)
    pages_found = sorted(list(set(h['page'] for h in hits)))

    # Printing the retrieved pages
    print(f"[Source Pages Retrieved: {pages_found}]")

    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)
    answer = ask_Qwen([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    return answer

# Test the function with a question from the PDF
print("Question:", question)
print("Answer:", rag_answer_with_source(question))

Question: What happens when crops are not rotated?
[Source Pages Retrieved: [9]]
Answer: When crops are not rotated, it can lead to several negative outcomes:

1. **Reduced Soil Productivity**: Without rotation, the soil may become depleted of nutrients over time due to continuous use of the same type of crop. This reduces its overall productivity.

2. **Increased Disease and Pest Problems**: Continuous planting of the same crop can create favorable conditions for specific diseases and pests to thrive. For example, monoculture corn fields are often plagued by rootworms because they have no natural predators to keep their populations under control.

3. **Decreased Biological Benefits**: Crop rotations help manage weeds through competition, mechanical methods, biological controls, or herbicides. When crops aren't rotated, these benefits diminish, making it harder to maintain healthy plants without additional inputs such as chemicals.

4. **Lower Yield Potential**: Different crops perform

In [53]:
test_questions = [
   "What is the recipe for baking chocolate chip cookies? ",
   "What safety rules should be observed for operating equipment?",
   "What is the capital city of France?",
   "Who is the sole owner/author of the guide published in January 2014?",
   "What are the benefits of rotating crops?",
]
for q in test_questions:
    print("Q:", q)
    print("A:", rag_answer(q), "\n")

Q: What is the recipe for baking chocolate chip cookies? 
A: I don't know, that is not in the document. 

Q: What safety rules should be observed for operating equipment?
A: According to the best farming practices mentioned in the context, all farming practices must maintain a balance between the economy, the environment, and health and safety. Specifically, when handling chemicals:

• Personal protective equipment (PPE) should always be worn.
• Equipment should be washed and disinfected before use.

These guidelines ensure that operations follow proper procedures while also prioritizing worker well-being and environmental protection. 

Q: What is the capital city of France?
A: I don't know, that is not in the document. 

Q: Who is the sole owner/author of the guide published in January 2014?
A: The information published in the foregoing guide does not engage ECHO; it is the sole responsibility of its author. 

Q: What are the benefits of rotating crops?
A: Rotating crops offers severa

| Question | Right page found? | Answer correct? | Invented anything? |
|---|---|---|---|
| 1. What are the benefits of rotating crops? | Page 9 | Yes | No |
| 2. What safety rules should be observed for operating equipment? | Page 9 | Yes | No |
| 3. Who is the sole owner/author of the guide published in January 2014? | Page 2 | Yes | No |
| 4. What is the capital city of France? | Not found | It doesn’t know | No |
| 5. What is the recipe for baking chocolate chip cookies? | Not found | It doesn’t know | No |

In [56]:
def chat(message, history):
    hits = retrieve(message)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)

    # Track which unique page numbers the context is pulled from
    pages_found = sorted(list(set(h['page'] for h in hits)))
    page_citation_string = ", ".join(f"page {p}" for p in pages_found) if pages_found else "None"

    messages = [{"role": "system", "content": SYSTEM_PROMPT + " Be polite. Refuse questions unrelated to the document."}]
    messages += history[-5:]                    # last few turns only, to keep it short and cheap
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})

    answer = ask_llm(messages)

    # Append the source page numbers clearly to the final chatbot reply
    if "I don't know" in answer or not hits:
        return answer
    else:
        return f"{answer}\n\n*(Information retrieved from: {page_citation_string})*"

bot = gr.ChatInterface(
    fn=chat,
    title="Best Farming Practices Assistant",
    examples=["What is this document about?"],
)
bot.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2cab0e04a81f3f3f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [57]:
%%writefile chatbot_app.py
import os
import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

PDF_PATH = "document.pdf"
Qwen_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document. "
    "Be polite. Refuse questions unrelated to the document."
)

Qwen = InferenceClient(api_key=os.environ["HF_TOKEN"])   # read from the Space secret
embedder = SentenceTransformer("all-MiniLM-L6-v2")

pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(PDF_PATH).pages)]
chunks = []
for page_num, text in pages:
    text = " ".join(text.split())
    for start in range(0, len(text), 400):
        piece = text[start:start + 500]
        if len(piece) > 50:
            chunks.append({"page": page_num, "text": piece})
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def retrieve(q, k=3):
    scores = vectors @ embedder.encode([q], normalize_embeddings=True)[0]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]

def chat(message, history):
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in retrieve(message))
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history[-6:]
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})
    r = Qwen.chat.completions.create(model=Qwen_MODEL, messages=messages, max_tokens=400, temperature=0.2)
    return r.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages", title="Best Farming Practices Assistant").launch()

Overwriting chatbot_app.py


In [58]:
%%writefile chatbot_requirements.txt
pypdf
sentence-transformers
huggingface_hub
numpy

Overwriting chatbot_requirements.txt
